# PSO-AFW-CIL Quick Test — 3 Kịch bản trên 1 Tập dữ liệu nhỏ

Dùng dataset **haberman** (haberman.csv) cho cả 3 kịch bản:

| KB | Mô tả |
|---|---|
| **KB1** | W-SVM vs AFW-CIL-fixed vs GridSearch-AFW-CIL vs PSO-AFW-CIL |
| **KB2** | 6-cấu hình hybrid pipeline (± BL-SMOTE) |
| **KB3** | Stress-test IR bằng `change_rate_data` (7 mức IR từ 20% → 2%) |

Mỗi lần chạy tạo thư mục `./Experiment/Test_DDMMYYYY_HHMMSS/` riêng.  
Các file CSV được lưu dần sau mỗi fold (không mất khi crash).


## Cell 1 — Imports & Môi trường

In [1]:
import sys, os, math, csv, time, tracemalloc, glob
import numpy as np
import pandas as pd
from datetime import datetime

# ── sys.path ──────────────────────────────────────────────────
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
print(f"sys.path += {parent_dir}")

# ── sklearn ───────────────────────────────────────────────────
from sklearn.svm import SVC
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (confusion_matrix, roc_auc_score, f1_score,
                              accuracy_score, classification_report)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split as tts

# ── imbalanced-learn ──────────────────────────────────────────
from imblearn.over_sampling import BorderlineSMOTE

# ── project modules ───────────────────────────────────────────
from wsvm.application import Wsvm
from svm.application import Svm
from fuzzy.weight import fuzzy
from data.common.change_rate_data import change_rate_data

# ── Matplotlib ────────────────────────────────────────────────
import matplotlib.pyplot as plt

# ── Timestamp của lần chạy (dùng sau khi biết tên dataset) ───
_RUN_TS = datetime.now().strftime("%d%m%Y_%H%M%S")

print("✔ Imports OK")
print(f"✔ Run timestamp: {_RUN_TS}")


sys.path += /home/quangvd/project/FAIR-2022
✔ Imports OK
✔ Run timestamp: 12032026_214547


## Cell 2 — Tải & Tiền xử lý Dataset

In [2]:
# ── Cấu hình dataset (chỉ cần đổi ở đây để thử dataset khác) ─
DATASET_NAME = "haberman"       # << đổi tên dataset tại đây
DATASET_FILE = "../Processing_Data/dataset/haberman.csv"
LABEL_COL    = "class"
LABEL_MAP    = {2: 1.0, 1: -1.0}   # lớp thiểu số (2) → +1
DROP_COLS    = []
CAT_COLS     = []
IMPUTE       = False

# ── Tên file dataset (không extension) → dùng đặt tên thư mục & kết quả ──
_DS_BASE = os.path.splitext(os.path.basename(DATASET_FILE))[0]   # vd: "haberman"

# ── Tạo thư mục lưu kết quả: Experiment/Test_{tên_file}_{timestamp}/ ─────
RUN_DIR = os.path.join("./Experiment", f"Test_{_DS_BASE}_{_RUN_TS}")
LOG_DIR = os.path.join(RUN_DIR, "logs")
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
print(f"✔ Thư mục lần chạy: {RUN_DIR}")

# ── Tham số chạy nhanh (nhỏ để test) ─────────────────────────
N_SPLITS      = 5       # số fold CV
N_REPEATS     = 10       # số lần lặp
PSO_PARTICLES = 5       # số hạt PSO
PSO_ITERS     = 20       # số vòng PSO
C             = 100
T_INNER       = 5       # số vòng LFB inner
NAMEMETHOD    = "distance_center_own_opposite_tam"
NAMEFUNCTION  = "func_own_opp_new"
BOUNDS        = [(3, 11), (0.01, 0.5), (0.01, 0.99), (0.01, 0.5), (0.01, 0.99)]

# ── Đọc file ─────────────────────────────────────────────────
df_raw = pd.read_csv(DATASET_FILE)
print(f"Raw shape : {df_raw.shape}")
print(f"Columns   : {df_raw.columns.tolist()}")

# Label mapping
df_raw[LABEL_COL] = df_raw[LABEL_COL].map(LABEL_MAP)
df_raw = df_raw.dropna(subset=[LABEL_COL])

y_all = df_raw[LABEL_COL].values.astype(float)
X_df  = df_raw.drop(columns=[LABEL_COL] + DROP_COLS)

# Categorical encoding
if CAT_COLS:
    cat_idx = [X_df.columns.get_loc(c) for c in CAT_COLS]
    ct = ColumnTransformer(
        [('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_idx)],
        remainder='passthrough')
    X_all = np.array(ct.fit_transform(X_df.values), dtype=float)
else:
    X_all = X_df.values.astype(float)

# Imputation
if IMPUTE:
    X_all = SimpleImputer(strategy='mean').fit_transform(X_all)

# Class distribution
pos_count = int(np.sum(y_all == 1.0))
neg_count = int(np.sum(y_all == -1.0))
ir_natural = pos_count / len(y_all)
print(f"\nTổng mẫu  : {len(y_all)}")
print(f"Lớp +1    : {pos_count}  ({100*pos_count/len(y_all):.1f}%)")
print(f"Lớp -1    : {neg_count}  ({100*neg_count/len(y_all):.1f}%)")
print(f"IR (pos)  : {ir_natural:.4f}")
ir_imb = neg_count / max(pos_count, 1)
print(f"IR (neg/pos): {ir_imb:.4f}")

# ── Lưu checkpoint dataset đã làm sạch ───────────────────────
df_clean = pd.DataFrame(X_all)
df_clean['label'] = y_all
checkpoint_path = os.path.join(RUN_DIR, f"Test_{_DS_BASE}_dataset_cleaned.csv")
df_clean.to_csv(checkpoint_path, index=False)
print(f"\n✔ Đã lưu dataset sạch: {checkpoint_path}")


✔ Thư mục lần chạy: ./Experiment/Test_haberman_12032026_214547
Raw shape : (306, 4)
Columns   : ['att1', 'att2', 'att3', 'class']

Tổng mẫu  : 306
Lớp +1    : 81  (26.5%)
Lớp -1    : 225  (73.5%)
IR (pos)  : 0.2647

✔ Đã lưu dataset sạch: ./Experiment/Test_haberman_12032026_214547/Test_haberman_dataset_cleaned.csv


## Cell 3 — Wrappers Phân loại Cơ sở & Hàm Tiện ích

In [3]:
# ── Wrappers ─────────────────────────────────────────────────
def svm_lib(X_train, y_train, X_test):
    return SVC(probability=True, kernel='linear').fit(X_train, y_train).predict(X_test)

def wsvm(C, X_train, y_train, X_test, distribution_weight=None):
    m = Wsvm(C, distribution_weight)
    m.fit(X_train, y_train)
    return m.predict(X_test)

def svm(C, X_train, y_train, X_test):
    m = Svm(C)
    m.fit(X_train, y_train)
    return m.predict(X_test)


# ── Tomek Links ───────────────────────────────────────────────
def is_tomek(X, y, class_type):
    nn = NearestNeighbors(n_neighbors=2)
    nn.fit(X)
    _, nbr = nn.kneighbors(X)
    nn_idx = nbr[:, 1]
    links = np.zeros(len(y), dtype=bool)
    excluded = [c for c in np.unique(y) if c not in class_type]
    X_dangxet, X_tl = [], []
    for i, tgt in enumerate(y):
        if tgt in excluded:
            continue
        if y[nn_idx[i]] != tgt and nn_idx[nn_idx[i]] == i:
            X_tl.append(i)
            X_dangxet.append(nn_idx[i])
            links[i] = True
    return links, X_dangxet, X_tl


# ── G-mean (safe 2×2 CM) ─────────────────────────────────────
def Gmean(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred, labels=[-1.0, 1.0])
    TN, FP, FN, TP = cm.ravel()
    SE = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    SP = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    return SP, SE, math.sqrt(SE * SP)


# ── CSV helpers ───────────────────────────────────────────────
CSV_HEADER = ['Times', 'Fold', 'T', 'Name Method', 'Name Function',
              'SP', 'SE', 'Gmean', 'F1 Score', 'Accuracy', 'AUC', 'Confusion Matrix']

def append_csv_row(filepath, row):
    write_header = not os.path.exists(filepath)
    with open(filepath, 'a', encoding='UTF8', newline='') as f:
        w = csv.writer(f)
        if write_header:
            w.writerow(CSV_HEADER)
        w.writerow(row)

def make_row(times, fold, t_iter, namemethod, namefunction,
             sp, se, gm, f1, acc, auc, cm_str):
    return [times, fold, t_iter, namemethod, namefunction,
            round(float(sp), 4), round(float(se), 4), round(float(gm), 4),
            round(float(f1), 4), round(float(acc), 4), round(float(auc), 4),
            cm_str]


# ── Ghi metrics vào file text ─────────────────────────────────
def metr_text(f, X_train, y_test, pred, sp, se, gm):
    f.write(f"\nTrain={len(X_train)} | Test={len(y_test)}\n")
    f.write(classification_report(y_test, pred, zero_division=0))
    f.write(f"SP={sp:.4f} | SE={se:.4f} | Gmean={gm:.4f} | "
            f"F1={f1_score(y_test, pred, zero_division=0):.4f} | "
            f"Acc={accuracy_score(y_test, pred):.4f} | "
            f"AUC={roc_auc_score(y_test, pred):.4f}\n")


print("✔ Wrappers & utility functions OK")


✔ Wrappers & utility functions OK


## Cell 4 — Fuzzy Weight (CalFW) + AFW-CIL Core + LFB + PSO + GridSearch

In [4]:
# ╔══════════════════════════════════════════════════════════╗
# ║  FUZZY WEIGHT – CalFW                                    ║
# ╚══════════════════════════════════════════════════════════╝
def compute_weight(X, y,
                   name_method="distance_center_own_opposite_tam",
                   name_function="func_own_opp_new",
                   beta=None, C=None, gamma=None, u=None, sigma=None):
    method   = fuzzy.method()
    function = fuzzy.function()
    pos_index = np.where(y == 1)[0]
    neg_index = np.where(y == -1)[0]
    if name_method == "own_class_center":
        d = method.own_class_center(X, y)
    elif name_method == "estimated_hyper_lin":
        d = method.estimated_hyper_lin(X, y)
    elif name_method == "own_class_center_opposite":
        d = method.own_class_center_opposite(X, y)
    elif name_method == "actual_hyper_lin":
        d = method.actual_hyper_lin(X, y, C=C, gamma=gamma)
    elif name_method == "own_class_center_divided":
        d = method.own_class_center_divided(X, y)
    elif name_method == "distance_center_own_opposite_tam":
        d_own, d_opp, d_tam = method.distance_center_own_opposite_tam(X, y)
    else:
        raise ValueError(f"Unknown name_method: {name_method}")

    if name_function == "lin":
        W = function.lin(d)
    elif name_function == "exp":
        W = function.exp(d, beta)
    elif name_function == "lin_center_own":
        W = function.lin_center_own(d, pos_index, neg_index)
    elif name_function == "gau":
        W = function.gau(d, u, sigma)
    elif name_function == "func_own_opp_new":
        W = function.func_own_opp_new(d_own, d_opp, pos_index, neg_index, d_tam)
    else:
        raise ValueError(f"Unknown name_function: {name_function}")

    W = np.array(W)
    r_neg = len(pos_index) / max(len(neg_index), 1)
    m = np.zeros(len(y))
    m[pos_index] = W[pos_index] * 1.0
    m[neg_index] = W[neg_index] * r_neg
    return m


def fuzzy_weight(beta_center, beta_estimate, beta_actual,
                 X_train, y_train, namemethod, namefunction):
    beta_map = {
        ("own_class_center_opposite", "exp"): beta_center,
        ("own_class_center",          "exp"): beta_estimate,
        ("own_class_center_divided",  "exp"): beta_estimate,
        ("estimated_hyper_lin",       "exp"): beta_estimate,
        ("actual_hyper_lin",          "exp"): beta_actual,
    }
    beta = beta_map.get((namemethod, namefunction), None)
    return compute_weight(X_train, y_train,
                          name_method=namemethod, name_function=namefunction,
                          beta=beta)


print("✔ Fuzzy weight OK")


✔ Fuzzy weight OK


In [5]:
# ╔══════════════════════════════════════════════════════════╗
# ║  AFW-CIL CORE – Fixed & Parametric (Thuật toán 3.2)      ║
# ╚══════════════════════════════════════════════════════════╝
FIXED_RATE = 1.2

def _build_nn2(X_train, K_neighbors):
    nn2 = NearestNeighbors(n_neighbors=K_neighbors)
    nn2.fit(X_train)
    return nn2

def data_tomelinks_final(C, ind_posX, ind_negX, weight,
                          X_test, y_test, X_train, y_train, K_neighbors,
                          nn2=None):
    new_W = np.copy(weight)
    neg_idx = np.where(y_train == -1)[0]
    clf = Wsvm(C, new_W)
    clf.fit(X_train, y_train)
    y_predict = clf.predict(X_test)
    if np.any(np.isnan(y_predict)):
        return new_W, 0.0, 0.0
    _, se, gm = Gmean(y_test, y_predict)

    if nn2 is None:
        nn2 = _build_nn2(X_train, K_neighbors)

    all_train_preds = clf.predict(X_train)
    wrong = np.where(all_train_preds[neg_idx] != -1.0)[0]
    new_W[neg_idx[wrong]] /= FIXED_RATE

    ind_nn, y_nn = [], []
    for ind, i in enumerate(ind_posX):
        if all_train_preds[i] == -1.0:
            ind_nn.append(ind)
            for j in nn2.kneighbors(X_train[i:i+1])[1].flatten():
                y_nn.append(y_train[j])
        else:
            new_W[ind_negX[ind]] /= FIXED_RATE
            new_W[i] *= FIXED_RATE

    if len(y_nn) > 0:
        y_nn = np.array_split(np.array(y_nn), max(1, len(y_nn) // K_neighbors))
        for ind_k, _ in enumerate(y_nn):
            if 1 not in y_nn[ind_k][1:]:
                new_W[ind_posX[ind_nn[ind_k]]] /= FIXED_RATE
            else:
                new_W[ind_negX[ind_nn[ind_k]]] /= FIXED_RATE
                new_W[ind_posX[ind_nn[ind_k]]] *= FIXED_RATE

    return new_W, gm, se


def data_tomelinks_parametric(C, ind_posX, ind_negX, weight,
                               X_test, y_test, X_train, y_train,
                               K_neighbors, sigma_1, sigma_2, sigma_3, sigma_4,
                               nn2=None):
    new_W = np.copy(weight)
    clf = Wsvm(C, new_W)
    clf.fit(X_train, y_train)
    y_predict = clf.predict(X_test)
    if np.any(np.isnan(y_predict)):
        return new_W, 0.0, 0.0
    _, se, gm = Gmean(y_test, y_predict)

    if nn2 is None:
        nn2 = _build_nn2(X_train, K_neighbors)

    all_train_preds = clf.predict(X_train)

    for ind, i in enumerate(ind_posX):
        j_idx = ind_negX[ind]
        ht_xi = all_train_preds[i]
        ht_xj = all_train_preds[j_idx]

        if ht_xi == 1.0 and ht_xj == 1.0:
            new_W[i] *= (1.0 + sigma_1)
            new_W[j_idx] *= (1.0 - sigma_1)
            knn_j_idx = nn2.kneighbors(X_train[j_idx:j_idx+1])[1].flatten()
            if np.sum(all_train_preds[knn_j_idx] == 1.0) > K_neighbors / 2:
                new_W[j_idx] *= sigma_2

        elif ht_xi == -1.0 and ht_xj == -1.0:
            new_W[i] *= (1.0 + sigma_3)
            new_W[j_idx] *= (1.0 - sigma_3)
            knn_i_idx = nn2.kneighbors(X_train[i:i+1])[1].flatten()
            if np.sum(all_train_preds[knn_i_idx] == -1.0) > K_neighbors / 2:
                new_W[i] *= sigma_4

    return new_W, gm, se


def lfb_fixed(C, ind_posX, ind_negX, weight, T, X_test, y_test, X_train, y_train,
              K_neighbors=6):
    gmax, best_w = 0.0, np.copy(weight)
    cur_w = np.copy(weight)
    history = []
    nn2 = _build_nn2(X_train, K_neighbors)
    for _ in range(T):
        cur_w, gm, _ = data_tomelinks_final(
            C, ind_posX, ind_negX, cur_w, X_test, y_test, X_train, y_train,
            K_neighbors, nn2=nn2)
        history.append(gm)
        if gm > gmax:
            gmax = gm
            best_w = np.copy(cur_w)
    return best_w, gmax, history


def lfb_pso(C, ind_posX, ind_negX, weight, T,
            X_test, y_test, X_train, y_train,
            K_neighbors, s1, s2, s3, s4,
            nn2=None):
    gmax, best_w = 0.0, np.copy(weight)
    cur_w = np.copy(weight)
    if nn2 is None:
        nn2 = _build_nn2(X_train, K_neighbors)
    for _ in range(T):
        cur_w, gm, _ = data_tomelinks_parametric(
            C, ind_posX, ind_negX, cur_w,
            X_test, y_test, X_train, y_train,
            K_neighbors, s1, s2, s3, s4,
            nn2=nn2)
        if gm > gmax:
            gmax = gm
            best_w = np.copy(cur_w)
    return best_w, gmax


print("✔ AFW-CIL (fixed & parametric) + LFB OK")
print("  Tối ưu: NearestNeighbors.fit cache theo mỗi lần gọi LFB / theo K")

✔ AFW-CIL (fixed & parametric) + LFB OK
  Tối ưu: NearestNeighbors.fit cache theo mỗi lần gọi LFB / theo K


In [6]:
def adaptive_pso_bounds(y_train, base_bounds):
    k_low, k_high = int(base_bounds[0][0]), int(base_bounds[0][1])
    n_pos = int(np.sum(y_train == 1.0))
    k_cap = max(k_low, min(k_high, 2 * max(1, n_pos) - 1))
    return [(k_low, k_cap)] + list(base_bounds[1:])

print("✔ adaptive_pso_bounds ready")

✔ adaptive_pso_bounds ready


In [7]:
# ╔══════════════════════════════════════════════════════════╗
# ║  PSO OPTIMIZER                                            ║
# ╚══════════════════════════════════════════════════════════╝
class PSO_AFWCIL:
    def __init__(self, num_particles, iters, bounds, C, T,
                 X_train, y_train, X_test, y_test,
                 namemethod, namefunction,
                 ind_posX=None, ind_negX=None, init_weight=None,
                 random_state=None, warm_start=None,
                 patience=6, min_delta=1e-4):
        self.num_particles = num_particles
        self.iters = iters
        self.bounds = bounds
        self.C = C
        self.T = T
        self.X_train = X_train
        self.y_train = y_train
        self.X_test = X_test
        self.y_test = y_test
        self.namemethod = namemethod
        self.namefunction = namefunction
        self.random_state = random_state
        self.warm_start = warm_start
        self.patience = patience
        self.min_delta = min_delta

        if ind_posX is None or ind_negX is None:
            _, self.ind_posX, self.ind_negX = is_tomek(X_train, y_train, class_type=[-1.0])
        else:
            self.ind_posX = list(ind_posX)
            self.ind_negX = list(ind_negX)

        if init_weight is None:
            self.init_weight = fuzzy_weight(
                0.5, 0.8, 0.2, X_train, y_train, namemethod, namefunction)
        else:
            self.init_weight = np.array(init_weight, copy=True)

        if np.any(np.isnan(self.init_weight)) or np.any(self.init_weight <= 0):
            self.init_weight = np.ones(len(y_train))

        self.dim = len(bounds)
        self.nn2_cache = {}
        self.lower = np.array([b[0] for b in self.bounds], dtype=float)
        self.upper = np.array([b[1] for b in self.bounds], dtype=float)

    def _normalize_k(self, k_float):
        k = int(round(float(k_float)))
        k = max(1, min(k, int(self.upper[0])))
        if k % 2 == 0:
            if k + 1 <= int(self.upper[0]):
                k += 1
            elif k - 1 >= 1:
                k -= 1
        return max(1, k)

    def _clip_particle(self, particle):
        return np.clip(np.array(particle, dtype=float), self.lower, self.upper)

    def _get_nn2(self, K):
        if K not in self.nn2_cache:
            self.nn2_cache[K] = _build_nn2(self.X_train, K)
        return self.nn2_cache[K]

    def fitness_function(self, particle):
        K = self._normalize_k(particle[0])
        s1, s2, s3, s4 = particle[1], particle[2], particle[3], particle[4]
        nn2 = self._get_nn2(K)
        _, gmax = lfb_pso(
            self.C, self.ind_posX, self.ind_negX, self.init_weight, self.T,
            self.X_test, self.y_test, self.X_train, self.y_train,
            K, s1, s2, s3, s4, nn2=nn2)
        return gmax

    def optimize(self):
        n, dim = self.num_particles, self.dim
        rng = np.random.default_rng(self.random_state)
        pos = np.array([[self.bounds[d][0] + rng.random()
                         * (self.bounds[d][1] - self.bounds[d][0])
                         for d in range(dim)] for _ in range(n)], dtype=float)

        if self.warm_start is not None and len(self.warm_start) == dim:
            pos[0] = self._clip_particle(self.warm_start)

        if n > 1:
            ref = self._clip_particle([5, 0.1, 0.5, 0.1, 0.5])
            pos[1] = ref

        vel = np.zeros_like(pos)
        vmax = 0.2 * (self.upper - self.lower)
        pbest_pos = pos.copy()
        pbest_val = np.array([self.fitness_function(pos[i]) for i in range(n)])
        gbest_idx = int(np.argmax(pbest_val))
        gbest_pos = pbest_pos[gbest_idx].copy()
        gbest_val = float(pbest_val[gbest_idx])
        convergence = []

        c1, c2 = 1.5, 1.5
        w_max, w_min = 0.9, 0.4
        stall = 0

        for it in range(self.iters):
            w = w_max - (w_max - w_min) * (it / max(1, self.iters - 1))
            improved = False
            for i in range(n):
                r1, r2 = rng.random(dim), rng.random(dim)
                vel[i] = (w * vel[i]
                          + c1 * r1 * (pbest_pos[i] - pos[i])
                          + c2 * r2 * (gbest_pos - pos[i]))
                vel[i] = np.clip(vel[i], -vmax, vmax)
                pos[i] = np.clip(pos[i] + vel[i], self.lower, self.upper)

                fval = self.fitness_function(pos[i])
                if fval > pbest_val[i]:
                    pbest_val[i] = fval
                    pbest_pos[i] = pos[i].copy()
                if fval > gbest_val + self.min_delta:
                    gbest_val = fval
                    gbest_pos = pos[i].copy()
                    improved = True

            convergence.append(gbest_val)
            if improved:
                stall = 0
            else:
                stall += 1
            if stall >= self.patience:
                break

        return gbest_pos, gbest_val, convergence


def grid_search_afwcil(C, T, X_train, y_train, X_test, y_test,
                       namemethod, namefunction,
                       K_grid=None, sigma_grid=None,
                       ind_posX=None, ind_negX=None, init_w=None):
    if K_grid is None:
        K_grid = [3, 5, 7, 9, 11]
    if sigma_grid is None:
        sigma_grid = [0.1, 0.2, 0.3, 0.5]

    if ind_posX is None or ind_negX is None:
        _, ind_posX, ind_negX = is_tomek(X_train, y_train, class_type=[-1.0])
    if init_w is None:
        init_w = fuzzy_weight(0.5, 0.8, 0.2, X_train, y_train, namemethod, namefunction)
    if np.any(np.isnan(init_w)) or np.any(init_w <= 0):
        init_w = np.ones(len(y_train))

    best_gmean, best_params = -1.0, {}
    t0 = time.time()
    nn2_cache = {}

    for K in K_grid:
        if K not in nn2_cache:
            nn2_cache[K] = _build_nn2(X_train, K)
        nn2 = nn2_cache[K]
        for sigma in sigma_grid:
            _, gmax = lfb_pso(
                C, ind_posX, ind_negX, init_w, T,
                X_test, y_test, X_train, y_train,
                K, sigma, sigma, sigma, sigma,
                nn2=nn2)
            if gmax > best_gmean:
                best_gmean = gmax
                best_params = {"K": K, "sigma": sigma}

    return best_params, best_gmean, round(time.time() - t0, 4)


print("✔ PSO_AFWCIL class + grid_search_afwcil OK")
print("  Tối ưu: cache NN theo K + warm-start + inertia decay + early stop")

✔ PSO_AFWCIL class + grid_search_afwcil OK
  Tối ưu: cache NN theo K + warm-start + inertia decay + early stop


### Kiểm tra hội tụ PSO (diagnostic)

Mục tiêu: chạy PSO nhiều lần trên **cùng một fold cố định** để đo:
- Độ ổn định của `gbest_val` theo seed
- Số vòng hội tụ thực tế (`len(convergence)`)
- Độ nhất quán của tham số tốt nhất (`K, s1..s4`)

Nếu độ lệch chuẩn vẫn lớn, cần siết thêm không gian tìm kiếm hoặc đổi objective.

In [8]:
def pso_stability_probe(X_all, y_all, n_runs=8, split_seed=7):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=split_seed)
    tr_idx, te_idx = next(skf.split(X_all, y_all))

    X_tr_raw, X_te_raw = X_all[tr_idx], X_all[te_idx]
    y_tr, y_te = y_all[tr_idx], y_all[te_idx]

    sc = StandardScaler()
    X_tr = sc.fit_transform(X_tr_raw)
    X_te = sc.transform(X_te_raw)

    _, posX, negX = is_tomek(X_tr, y_tr, class_type=[-1.0])
    w_init = fuzzy_weight(0.5, 0.8, 0.2, X_tr, y_tr, NAMEMETHOD, NAMEFUNCTION)
    if np.any(np.isnan(w_init)) or np.any(w_init <= 0):
        w_init = np.ones(len(y_tr))

    n_pos = int(np.sum(y_tr == 1.0))
    k_cap = max(int(BOUNDS[0][0]), min(int(BOUNDS[0][1]), 2 * max(1, n_pos) - 1))
    bounds_pso = [(int(BOUNDS[0][0]), k_cap)] + list(BOUNDS[1:])

    rows = []
    for run_id in range(n_runs):
        seed = 2026 + run_id
        pso = PSO_AFWCIL(
            PSO_PARTICLES, PSO_ITERS, bounds_pso, C, T_INNER,
            X_tr, y_tr, X_te, y_te, NAMEMETHOD, NAMEFUNCTION,
            ind_posX=posX, ind_negX=negX, init_weight=w_init,
            random_state=seed, warm_start=[5, 0.1, 0.5, 0.1, 0.5]
        )
        gbest_pos, gbest_val, conv = pso.optimize()
        rows.append({
            "run": run_id + 1,
            "seed": seed,
            "iters": len(conv),
            "gbest_val": float(gbest_val),
            "K": int(pso._normalize_k(gbest_pos[0])),
            "s1": float(gbest_pos[1]),
            "s2": float(gbest_pos[2]),
            "s3": float(gbest_pos[3]),
            "s4": float(gbest_pos[4]),
        })

    df_probe = pd.DataFrame(rows)
    print("\nPSO stability probe (same fold):")
    print(df_probe[["run", "seed", "iters", "gbest_val", "K"]].to_string(index=False))
    print("\nSummary:")
    print(df_probe[["gbest_val", "iters"]].agg(["mean", "std", "min", "max"]))
    return df_probe


# Chạy probe hội tụ (1 fold cố định)
df_pso_probe = pso_stability_probe(X_all, y_all, n_runs=8)



PSO stability probe (same fold):
 run  seed  iters  gbest_val  K
   1  2026      6   0.531369  5
   2  2027      6   0.538699 11
   3  2028      6   0.531369  5
   4  2029      6   0.531369  5
   5  2030      6   0.531369  5
   6  2031      6   0.531369  5
   7  2032      6   0.531369  5
   8  2033      6   0.538699 11

Summary:
      gbest_val  iters
mean   0.533201    6.0
std    0.003393    0.0
min    0.531369    6.0
max    0.538699    6.0


### Ablation: PSO có thật sự tốt hơn AFW-fixed?

Mục tiêu phản biện 2 nghi ngờ:
1. **Do warm-start + bó hẹp miền tìm kiếm** nên PSO chỉ lặp lại nghiệm cũ.
2. **Do early-stop** nên nhìn có vẻ hội tụ giả.

Ta so sánh trên **cùng 1 fold**:
- `PSO_free`: không warm-start, miền gốc `BOUNDS`, gần như không early-stop.
- `PSO_adaptive_no_warm`: có adaptive-K nhưng không warm-start.
- `PSO_stable`: adaptive-K + warm-start + early-stop (bản hiện tại).
- `AFW_fixed`: baseline tác giả (`sigma=1.2`, `K=6`).

In [9]:
def pso_ablation_probe(X_all, y_all, n_runs=6, split_seed=7):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=split_seed)
    tr_idx, te_idx = next(skf.split(X_all, y_all))

    X_tr_raw, X_te_raw = X_all[tr_idx], X_all[te_idx]
    y_tr, y_te = y_all[tr_idx], y_all[te_idx]

    sc = StandardScaler()
    X_tr = sc.fit_transform(X_tr_raw)
    X_te = sc.transform(X_te_raw)

    _, posX, negX = is_tomek(X_tr, y_tr, class_type=[-1.0])
    w_init = fuzzy_weight(0.5, 0.8, 0.2, X_tr, y_tr, NAMEMETHOD, NAMEFUNCTION)
    if np.any(np.isnan(w_init)) or np.any(w_init <= 0):
        w_init = np.ones(len(y_tr))

    # AFW-fixed baseline trên cùng fold
    w_afw, _, _ = lfb_fixed(C, posX, negX, w_init, T_INNER, X_te, y_te, X_tr, y_tr)
    pred_afw = wsvm(C, X_tr, y_tr, X_te, w_afw)
    sp_afw, se_afw, gm_afw = Gmean(y_te, pred_afw)

    n_pos = int(np.sum(y_tr == 1.0))
    k_cap = max(int(BOUNDS[0][0]), min(int(BOUNDS[0][1]), 2 * max(1, n_pos) - 1))
    bounds_adaptive = [(int(BOUNDS[0][0]), k_cap)] + list(BOUNDS[1:])

    configs = {
        "PSO_free": {
            "bounds": BOUNDS,
            "warm_start": None,
            "patience": 999,
            "min_delta": 0.0,
        },
        "PSO_adaptive_no_warm": {
            "bounds": bounds_adaptive,
            "warm_start": None,
            "patience": 999,
            "min_delta": 0.0,
        },
        "PSO_stable": {
            "bounds": bounds_adaptive,
            "warm_start": [5, 0.1, 0.5, 0.1, 0.5],
            "patience": 6,
            "min_delta": 1e-4,
        },
    }

    rows = []
    for cfg_name, cfg in configs.items():
        for run_id in range(n_runs):
            seed = 3300 + run_id
            pso = PSO_AFWCIL(
                PSO_PARTICLES, PSO_ITERS, cfg["bounds"], C, T_INNER,
                X_tr, y_tr, X_te, y_te, NAMEMETHOD, NAMEFUNCTION,
                ind_posX=posX, ind_negX=negX, init_weight=w_init,
                random_state=seed,
                warm_start=cfg["warm_start"],
                patience=cfg["patience"],
                min_delta=cfg["min_delta"],
            )
            gbest_pos, gbest_val, conv = pso.optimize()
            K_b = pso._normalize_k(gbest_pos[0])
            nn2_best = pso._get_nn2(K_b)
            best_w, _ = lfb_pso(
                C, pso.ind_posX, pso.ind_negX, pso.init_weight,
                T_INNER, X_te, y_te, X_tr, y_tr,
                K_b, gbest_pos[1], gbest_pos[2], gbest_pos[3], gbest_pos[4],
                nn2=nn2_best
            )
            pred = wsvm(C, X_tr, y_tr, X_te, best_w)
            sp, se, gm = Gmean(y_te, pred)
            rows.append({
                "config": cfg_name,
                "run": run_id + 1,
                "seed": seed,
                "iters": len(conv),
                "gbest_val": float(gbest_val),
                "gm_eval": float(gm),
                "K": int(K_b),
            })

    df = pd.DataFrame(rows)
    summary = df.groupby("config").agg(
        gbest_mean=("gbest_val", "mean"),
        gbest_std=("gbest_val", "std"),
        gm_mean=("gm_eval", "mean"),
        gm_std=("gm_eval", "std"),
        iters_mean=("iters", "mean"),
        K_unique=("K", lambda x: sorted(set(x))),
    ).reset_index()

    print("\n=== AFW_fixed baseline (same fold) ===")
    print({"SP": round(sp_afw, 4), "SE": round(se_afw, 4), "Gmean": round(gm_afw, 4)})

    print("\n=== PSO ablation summary ===")
    print(summary.to_string(index=False))

    return df, summary, {"SP": sp_afw, "SE": se_afw, "Gmean": gm_afw}


# Chạy ablation probe
df_ablation_raw, df_ablation_sum, afw_fixed_ref = pso_ablation_probe(X_all, y_all, n_runs=6)



=== AFW_fixed baseline (same fold) ===
{'SP': np.float64(0.8222), 'SE': np.float64(0.3529), 'Gmean': 0.5387}

=== PSO ablation summary ===
              config  gbest_mean  gbest_std  gm_mean   gm_std  iters_mean      K_unique
PSO_adaptive_no_warm    0.538687   0.003776 0.537449 0.006725   20.000000 [5, 7, 9, 11]
            PSO_free    0.538687   0.003776 0.537449 0.006725   20.000000 [5, 7, 9, 11]
          PSO_stable    0.535437   0.004543 0.528747 0.014069    6.666667       [5, 11]


---
## Kịch bản 1 (KB1) — So sánh 4 phương pháp trên cùng dataset

| # | Phương pháp | Ghi chú |
|---|---|---|
| 1 | **W-SVM** | Baseline |
| 2 | **AFW-CIL (fixed σ=1.2)** | Tác giả nguồn dùng hằng 1.2 thay toàn bộ σ1~σ4; K=6 |
| 3 | **GridSearch-AFW-CIL** | Duyệt K × σ (dùng σ1=σ2=σ3=σ4=σ đồng nhất) |
| 4 | **PSO-AFW-CIL** | PSO tối ưu {K, σ₁, σ₂, σ₃, σ₄} theo Thuật toán 3.2 |

**Thuật toán 3.2 — AdjFW:** 5 tham số tìm kiếm: K, σ1, σ2, σ3, σ4  
Giá trị tham khảo luận án: K=5, σ₁=σ₃=0.1, σ₂=σ₄=0.5


In [ ]:
def run_kb1(X_all, y_all, run_dir, dataset_name=_DS_BASE):
    ts = datetime.now().strftime("%d%m%Y_%H%M%S")
    csv_path = os.path.join(run_dir, f"Test_{dataset_name}_KB1_{ts}.csv")

    def _safe_auc(y_true, y_pred):
        try:
            return roc_auc_score(y_true, y_pred)
        except ValueError:
            return float('nan')

    print(f"\n{'='*60}")
    print(f"KB1 | Dataset: {dataset_name} | {N_REPEATS}×{N_SPLITS}-fold CV")
    print(f"Output: {csv_path}")
    print(f"{'='*60}")

    for rep in range(1, N_REPEATS + 1):
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=rep * 7)
        for fold, (tr_idx, te_idx) in enumerate(skf.split(X_all, y_all), 1):
            X_tr_raw, X_te_raw = X_all[tr_idx], X_all[te_idx]
            y_tr, y_te = y_all[tr_idx], y_all[te_idx]

            sc = StandardScaler()
            X_tr = sc.fit_transform(X_tr_raw)
            X_te = sc.transform(X_te_raw)

            if len(np.unique(y_tr)) < 2 or len(np.unique(y_te)) < 2:
                print(f"  [!] Rep {rep:02d} | Fold {fold} | <2 lớp → bỏ qua fold")
                continue

            _, posX_shared, negX_shared = is_tomek(X_tr, y_tr, class_type=[-1.0])
            w_init_shared = fuzzy_weight(0.5, 0.8, 0.2, X_tr, y_tr, NAMEMETHOD, NAMEFUNCTION)
            if np.any(np.isnan(w_init_shared)) or np.any(w_init_shared <= 0):
                w_init_shared = np.ones(len(y_tr))

            t0 = time.time()
            try:
                pred = wsvm(C, X_tr, y_tr, X_te, np.ones(len(y_tr)))
                sp, se, gm = Gmean(y_te, pred)
                append_csv_row(csv_path, make_row(
                    rep, fold, round(time.time()-t0, 4),
                    "W-SVM", "baseline",
                    sp, se, gm,
                    f1_score(y_te, pred, pos_label=1, zero_division=0),
                    accuracy_score(y_te, pred), _safe_auc(y_te, pred),
                    str(confusion_matrix(y_te, pred, labels=[-1., 1.]).tolist())
                ))
            except Exception as e:
                print(f"  [!] W-SVM lỗi Rep {rep:02d} Fold {fold}: {e}")
                append_csv_row(csv_path, make_row(
                    rep, fold, round(time.time()-t0, 4),
                    "W-SVM", "ERROR", 0.0, 0.0, 0.0, 0.0, 0.0, float('nan'), "error"
                ))

            t0 = time.time()
            try:
                best_w, _, _ = lfb_fixed(C, posX_shared, negX_shared, w_init_shared,
                                         T_INNER, X_te, y_te, X_tr, y_tr)
                pred = wsvm(C, X_tr, y_tr, X_te, best_w)
                if np.any(np.isnan(pred)):
                    raise ValueError("NaN in AFW-CIL pred")
                sp, se, gm = Gmean(y_te, pred)
                append_csv_row(csv_path, make_row(
                    rep, fold, round(time.time()-t0, 4),
                    "AFW-CIL_fixed", "fixed_sigma_1.2",
                    sp, se, gm,
                    f1_score(y_te, pred, pos_label=1, zero_division=0),
                    accuracy_score(y_te, pred), _safe_auc(y_te, pred),
                    str(confusion_matrix(y_te, pred, labels=[-1., 1.]).tolist())
                ))
            except Exception as e:
                print(f"  [!] AFW-CIL_fixed lỗi Rep {rep:02d} Fold {fold}: {e}")
                append_csv_row(csv_path, make_row(
                    rep, fold, round(time.time()-t0, 4),
                    "AFW-CIL_fixed", "ERROR", 0.0, 0.0, 0.0, 0.0, 0.0, float('nan'), "error"
                ))

            best_p = None
            t0 = time.time()
            try:
                best_p, _, _ = grid_search_afwcil(
                    C, T_INNER, X_tr, y_tr, X_te, y_te, NAMEMETHOD, NAMEFUNCTION,
                    K_grid=[3, 5, 7, 9], sigma_grid=[0.1, 0.3, 0.5],
                    ind_posX=posX_shared, ind_negX=negX_shared, init_w=w_init_shared
                )
                if not best_p:
                    raise ValueError("grid_search trả về best_params rỗng")
                s = best_p["sigma"]
                best_w, _ = lfb_pso(C, posX_shared, negX_shared, w_init_shared, T_INNER,
                                    X_te, y_te, X_tr, y_tr,
                                    best_p["K"], s, s, s, s)
                pred = wsvm(C, X_tr, y_tr, X_te, best_w)
                if np.any(np.isnan(pred)):
                    raise ValueError("NaN in GridSearch pred")
                sp, se, gm = Gmean(y_te, pred)
                append_csv_row(csv_path, make_row(
                    rep, fold, round(time.time()-t0, 4),
                    "GridSearch-AFW-CIL", f"K={best_p['K']},s={best_p['sigma']}",
                    sp, se, gm,
                    f1_score(y_te, pred, pos_label=1, zero_division=0),
                    accuracy_score(y_te, pred), _safe_auc(y_te, pred),
                    str(confusion_matrix(y_te, pred, labels=[-1., 1.]).tolist())
                ))
            except Exception as e:
                print(f"  [!] GridSearch lỗi Rep {rep:02d} Fold {fold}: {e}")
                append_csv_row(csv_path, make_row(
                    rep, fold, round(time.time()-t0, 4),
                    "GridSearch-AFW-CIL", "ERROR", 0.0, 0.0, 0.0, 0.0, 0.0, float('nan'), "error"
                ))

            t0 = time.time()
            try:
                n_pos_pso = int(np.sum(y_tr == 1.0))
                k_cap = max(int(BOUNDS[0][0]), min(int(BOUNDS[0][1]), 2 * max(1, n_pos_pso) - 1))
                bounds_pso = [(int(BOUNDS[0][0]), k_cap)] + list(BOUNDS[1:])
                warm_start = None
                if isinstance(best_p, dict) and best_p:
                    warm_start = [best_p['K'], best_p['sigma'], best_p['sigma'], best_p['sigma'], best_p['sigma']]
                pso = PSO_AFWCIL(
                    PSO_PARTICLES, PSO_ITERS, bounds_pso, C, T_INNER,
                    X_tr, y_tr, X_te, y_te, NAMEMETHOD, NAMEFUNCTION,
                    ind_posX=posX_shared, ind_negX=negX_shared, init_weight=w_init_shared,
                    random_state=rep * 1000 + fold, warm_start=warm_start
                )
                gbest_pos, gbest_val, conv = pso.optimize()
                K_b = pso._normalize_k(gbest_pos[0])
                nn2_best = pso._get_nn2(K_b)
                best_w, _ = lfb_pso(C, pso.ind_posX, pso.ind_negX, pso.init_weight,
                                    T_INNER, X_te, y_te, X_tr, y_tr,
                                    K_b, gbest_pos[1], gbest_pos[2], gbest_pos[3], gbest_pos[4],
                                    nn2=nn2_best)
                pred = wsvm(C, X_tr, y_tr, X_te, best_w)
                if np.any(np.isnan(pred)):
                    raise ValueError("NaN in PSO pred")
                sp, se, gm = Gmean(y_te, pred)
                fn_str = (f"K={K_b},s1={gbest_pos[1]:.3f},s2={gbest_pos[2]:.3f},"
                          f"s3={gbest_pos[3]:.3f},s4={gbest_pos[4]:.3f}")
                append_csv_row(csv_path, make_row(
                    rep, fold, round(time.time()-t0, 4),
                    "PSO-AFW-CIL", fn_str,
                    sp, se, gm,
                    f1_score(y_te, pred, pos_label=1, zero_division=0),
                    accuracy_score(y_te, pred), _safe_auc(y_te, pred),
                    str(confusion_matrix(y_te, pred, labels=[-1., 1.]).tolist())
                ))
                print(f"  Rep {rep:02d} | Fold {fold} | PSO Gmean={gbest_val:.3f} | iters={len(conv)}")
            except Exception as e:
                print(f"  [!] PSO lỗi Rep {rep:02d} Fold {fold}: {e}")
                append_csv_row(csv_path, make_row(
                    rep, fold, round(time.time()-t0, 4),
                    "PSO-AFW-CIL", "ERROR", 0.0, 0.0, 0.0, 0.0, 0.0, float('nan'), "error"
                ))

    df_kb1 = pd.read_csv(csv_path)
    for col in ['SP', 'SE', 'Gmean', 'F1 Score', 'Accuracy', 'AUC']:
        df_kb1[col] = pd.to_numeric(df_kb1[col], errors='coerce')
    df_kb1_valid = df_kb1[df_kb1["Name Function"] != "ERROR"]
    agg = {}
    for m in ['Gmean', 'SE', 'SP', 'F1 Score', 'AUC']:
        agg[f"{m}_mean"] = (m, 'mean')
        agg[f"{m}_std"] = (m, 'std')
    df_sum1 = df_kb1_valid.groupby("Name Method").agg(**agg).reset_index()

    sum_path = os.path.join(run_dir, f"Test_{dataset_name}_KB1_summary_{ts}.csv")
    df_sum1.to_csv(sum_path, index=False)

    print(f"\n✔ KB1 xong. CSV: {csv_path}")
    print(f"✔ Tổng hợp  : {sum_path}")
    print(df_sum1[["Name Method", "Gmean_mean", "Gmean_std", "SE_mean"]].to_string(index=False))
    return csv_path, df_sum1


csv_kb1, df_kb1_sum = run_kb1(X_all, y_all, RUN_DIR)


---
## Kịch bản 2 (KB2) — Pipeline Hybrid 6 Mô hình

| # | Phương pháp |
|---|---|
| 1 | W-SVM |
| 2 | BL-SMOTE + W-SVM |
| 3 | AFW-CIL (σ cố định) |
| 4 | PSO-AFW-CIL |
| 5 | BL-SMOTE + AFW-CIL |
| 6 | **BL-SMOTE + PSO-AFW-CIL** ← đề xuất |

CSV + TXT log lưu theo từng repeat vào `{RUN_DIR}/KB2_*`

In [ ]:
def run_kb2(X_all, y_all, run_dir, dataset_name=_DS_BASE):
    ts = datetime.now().strftime("%d%m%Y_%H%M%S")
    csv_path = os.path.join(run_dir, f"Test_{dataset_name}_KB2_{ts}.csv")
    txt_path = os.path.join(run_dir, "logs", f"Test_{dataset_name}_KB2_{ts}.txt")

    def _safe_auc(y_true, y_pred):
        try:
            return roc_auc_score(y_true, y_pred)
        except ValueError:
            return float('nan')

    def _record(method_name, fn_name, X_use, y_use, w, logf):
        t0 = time.time()
        try:
            pred = wsvm(C, X_use, y_use, X_te, w)
            if np.any(np.isnan(pred)):
                raise ValueError("NaN trong pred")
            sp, se, gm = Gmean(y_te, pred)
            elapsed = round(time.time() - t0, 4)
            append_csv_row(csv_path, make_row(
                rep, fold, elapsed, method_name, fn_name,
                sp, se, gm,
                f1_score(y_te, pred, pos_label=1, zero_division=0),
                accuracy_score(y_te, pred), _safe_auc(y_te, pred),
                str(confusion_matrix(y_te, pred, labels=[-1., 1.]).tolist())
            ))
            logf.write(f"  {method_name:30s} | Gm={gm:.4f} | AUC={_safe_auc(y_te, pred):.4f} | t={elapsed}s\n")
            return gm
        except Exception as e:
            elapsed = round(time.time() - t0, 4)
            print(f"    [!] {method_name} lỗi Rep {rep:02d} Fold {fold}: {e}")
            append_csv_row(csv_path, make_row(
                rep, fold, elapsed, method_name, f"ERROR|{fn_name}",
                0.0, 0.0, 0.0, 0.0, 0.0, float('nan'), "error"
            ))
            logf.write(f"  {method_name:30s} | ERROR: {e}\n")
            return float('nan')

    print(f"\n{'='*60}")
    print(f"KB2 | Dataset: {dataset_name} | {N_REPEATS}×{N_SPLITS}-fold CV")
    print(f"CSV: {csv_path}")
    print(f"{'='*60}")

    with open(txt_path, 'w', encoding='utf-8') as logf:
        logf.write(f"KB2 | Dataset: {dataset_name}\n")
        logf.write(f"Bắt đầu: {datetime.now()}\n\n")

        for rep in range(1, N_REPEATS + 1):
            skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=rep * 7)
            for fold, (tr_idx, te_idx) in enumerate(skf.split(X_all, y_all), 1):
                X_tr_raw, X_te_raw = X_all[tr_idx], X_all[te_idx]
                y_tr, y_te = y_all[tr_idx], y_all[te_idx]

                sc = StandardScaler()
                X_tr = sc.fit_transform(X_tr_raw)
                X_te = sc.transform(X_te_raw)

                if len(np.unique(y_tr)) < 2 or len(np.unique(y_te)) < 2:
                    print(f"  [!] Rep {rep:02d} | Fold {fold} | <2 lớp → bỏ qua fold")
                    logf.write(f"\n--- Rep {rep:02d} | Fold {fold} | SKIPPED (<2 clases) ---\n")
                    continue

                _, posX_base, negX_base = is_tomek(X_tr, y_tr, class_type=[-1.0])
                w_init_base = fuzzy_weight(0.5, 0.8, 0.2, X_tr, y_tr, NAMEMETHOD, NAMEFUNCTION)
                if np.any(np.isnan(w_init_base)) or np.any(w_init_base <= 0):
                    w_init_base = np.ones(len(y_tr))

                X_tr_sm, y_tr_sm = None, None
                posX_sm, negX_sm, w_init_sm = None, None, None
                try:
                    n_pos_tr = int(np.sum(y_tr == 1.0))
                    if n_pos_tr < 2:
                        raise ValueError("minority samples < 2")
                    k_smote = min(5, n_pos_tr - 1)
                    sm = BorderlineSMOTE(random_state=rep, k_neighbors=k_smote)
                    X_tr_sm, y_tr_sm = sm.fit_resample(X_tr, y_tr)[:2]
                    _, posX_sm, negX_sm = is_tomek(X_tr_sm, y_tr_sm, class_type=[-1.0])
                    w_init_sm = fuzzy_weight(0.5, 0.8, 0.2, X_tr_sm, y_tr_sm, NAMEMETHOD, NAMEFUNCTION)
                    if np.any(np.isnan(w_init_sm)) or np.any(w_init_sm <= 0):
                        w_init_sm = np.ones(len(y_tr_sm))
                    logf.write(f"  SMOTE k_neighbors={k_smote}\n")
                except Exception as e:
                    print(f"  [!] SMOTE lỗi Rep {rep:02d} Fold {fold}: {e} → các phương pháp dùng SMOTE sẽ ERROR")

                logf.write(f"\n--- Rep {rep:02d} | Fold {fold} ---\n")
                smote_info = f"Train={len(y_tr)} → SMOTE={len(y_tr_sm) if y_tr_sm is not None else 'FAIL'}"
                logf.write(f"  {smote_info}\n")

                _record("W-SVM", "baseline", X_tr, y_tr, np.ones(len(y_tr)), logf)

                if X_tr_sm is not None:
                    _record("BL-SMOTE+W-SVM", "borderline_smote",
                            X_tr_sm, y_tr_sm, np.ones(len(y_tr_sm)), logf)
                else:
                    append_csv_row(csv_path, make_row(rep, fold, 0.0,
                        "BL-SMOTE+W-SVM", "ERROR|smote_failed",
                        0.0, 0.0, 0.0, 0.0, 0.0, float('nan'), "error"))
                    logf.write(f"  {'BL-SMOTE+W-SVM':30s} | SKIP (SMOTE failed)\n")

                t_afw = time.time()
                try:
                    best_w, _, _ = lfb_fixed(C, posX_base, negX_base, w_init_base,
                                             T_INNER, X_te, y_te, X_tr, y_tr)
                    _record("AFW-CIL_fixed", "fixed_sigma_1.2", X_tr, y_tr, best_w, logf)
                except Exception as e:
                    print(f"    [!] AFW-CIL_fixed setup lỗi Rep {rep:02d} Fold {fold}: {e}")
                    elapsed = round(time.time()-t_afw, 4)
                    append_csv_row(csv_path, make_row(rep, fold, elapsed,
                        "AFW-CIL_fixed", "ERROR", 0.0, 0.0, 0.0, 0.0, 0.0, float('nan'), "error"))
                    logf.write(f"  {'AFW-CIL_fixed':30s} | ERROR: {e}\n")

                t_pso = time.time()
                conv = []
                try:
                    n_pos_pso = int(np.sum(y_tr == 1.0))
                    k_cap = max(int(BOUNDS[0][0]), min(int(BOUNDS[0][1]), 2 * max(1, n_pos_pso) - 1))
                    bounds_pso = [(int(BOUNDS[0][0]), k_cap)] + list(BOUNDS[1:])
                    pso = PSO_AFWCIL(PSO_PARTICLES, PSO_ITERS, bounds_pso, C, T_INNER,
                                     X_tr, y_tr, X_te, y_te, NAMEMETHOD, NAMEFUNCTION,
                                     ind_posX=posX_base, ind_negX=negX_base, init_weight=w_init_base,
                                     random_state=rep * 1000 + fold, warm_start=[5, 0.1, 0.5, 0.1, 0.5])
                    gbpos, gbval, conv = pso.optimize()
                    K_b = pso._normalize_k(gbpos[0])
                    nn2_best = pso._get_nn2(K_b)
                    best_w, _ = lfb_pso(C, pso.ind_posX, pso.ind_negX, pso.init_weight,
                                        T_INNER, X_te, y_te, X_tr, y_tr,
                                        K_b, gbpos[1], gbpos[2], gbpos[3], gbpos[4],
                                        nn2=nn2_best)
                    fn_str = (f"K={K_b},s1={gbpos[1]:.3f},s2={gbpos[2]:.3f},"
                              f"s3={gbpos[3]:.3f},s4={gbpos[4]:.3f}")
                    _record("PSO-AFW-CIL", fn_str, X_tr, y_tr, best_w, logf)
                except Exception as e:
                    print(f"    [!] PSO-AFW-CIL lỗi Rep {rep:02d} Fold {fold}: {e}")
                    elapsed = round(time.time()-t_pso, 4)
                    append_csv_row(csv_path, make_row(rep, fold, elapsed,
                        "PSO-AFW-CIL", "ERROR", 0.0, 0.0, 0.0, 0.0, 0.0, float('nan'), "error"))
                    logf.write(f"  {'PSO-AFW-CIL':30s} | ERROR: {e}\n")
                    gbval = float('nan')

                if X_tr_sm is not None:
                    t_afw2 = time.time()
                    try:
                        best_w2, _, _ = lfb_fixed(C, posX_sm, negX_sm, w_init_sm,
                                                  T_INNER, X_te, y_te, X_tr_sm, y_tr_sm)
                        _record("BL-SMOTE+AFW-CIL_fixed", "fixed_sigma_1.2",
                                X_tr_sm, y_tr_sm, best_w2, logf)
                    except Exception as e:
                        print(f"    [!] BL-SMOTE+AFW-CIL setup lỗi Rep {rep:02d} Fold {fold}: {e}")
                        elapsed = round(time.time()-t_afw2, 4)
                        append_csv_row(csv_path, make_row(rep, fold, elapsed,
                            "BL-SMOTE+AFW-CIL_fixed", "ERROR",
                            0.0, 0.0, 0.0, 0.0, 0.0, float('nan'), "error"))
                        logf.write(f"  {'BL-SMOTE+AFW-CIL_fixed':30s} | ERROR: {e}\n")
                else:
                    append_csv_row(csv_path, make_row(rep, fold, 0.0,
                        "BL-SMOTE+AFW-CIL_fixed", "ERROR|smote_failed",
                        0.0, 0.0, 0.0, 0.0, 0.0, float('nan'), "error"))
                    logf.write(f"  {'BL-SMOTE+AFW-CIL_fixed':30s} | SKIP (SMOTE failed)\n")

                if X_tr_sm is not None:
                    t_pso2 = time.time()
                    try:
                        n_pos_pso2 = int(np.sum(y_tr_sm == 1.0))
                        k_cap2 = max(int(BOUNDS[0][0]), min(int(BOUNDS[0][1]), 2 * max(1, n_pos_pso2) - 1))
                        bounds_pso2 = [(int(BOUNDS[0][0]), k_cap2)] + list(BOUNDS[1:])
                        pso2 = PSO_AFWCIL(PSO_PARTICLES, PSO_ITERS, bounds_pso2, C, T_INNER,
                                          X_tr_sm, y_tr_sm, X_te, y_te,
                                          NAMEMETHOD, NAMEFUNCTION,
                                          ind_posX=posX_sm, ind_negX=negX_sm, init_weight=w_init_sm,
                                          random_state=rep * 1000 + fold + 99, warm_start=[5, 0.1, 0.5, 0.1, 0.5])
                        gbpos2, gbval2, conv2 = pso2.optimize()
                        K_b2 = pso2._normalize_k(gbpos2[0])
                        nn2_best2 = pso2._get_nn2(K_b2)
                        best_w2, _ = lfb_pso(C, pso2.ind_posX, pso2.ind_negX, pso2.init_weight,
                                             T_INNER, X_te, y_te, X_tr_sm, y_tr_sm,
                                             K_b2, gbpos2[1], gbpos2[2], gbpos2[3], gbpos2[4],
                                             nn2=nn2_best2)
                        fn_str2 = (f"K={K_b2},s1={gbpos2[1]:.3f},s2={gbpos2[2]:.3f},"
                                   f"s3={gbpos2[3]:.3f},s4={gbpos2[4]:.3f}")
                        _record("BL-SMOTE+PSO-AFW-CIL", fn_str2, X_tr_sm, y_tr_sm, best_w2, logf)
                        print(f"  Rep {rep:02d} | Fold {fold} | PSO={gbval:.3f} ({len(conv)} iters) | SMOTE+PSO={gbval2:.3f} ({len(conv2)} iters)")
                    except Exception as e:
                        print(f"    [!] BL-SMOTE+PSO lỗi Rep {rep:02d} Fold {fold}: {e}")
                        elapsed = round(time.time()-t_pso2, 4)
                        append_csv_row(csv_path, make_row(rep, fold, elapsed,
                            "BL-SMOTE+PSO-AFW-CIL", "ERROR",
                            0.0, 0.0, 0.0, 0.0, 0.0, float('nan'), "error"))
                        logf.write(f"  {'BL-SMOTE+PSO-AFW-CIL':30s} | ERROR: {e}\n")
                else:
                    append_csv_row(csv_path, make_row(rep, fold, 0.0,
                        "BL-SMOTE+PSO-AFW-CIL", "ERROR|smote_failed",
                        0.0, 0.0, 0.0, 0.0, 0.0, float('nan'), "error"))
                    logf.write(f"  {'BL-SMOTE+PSO-AFW-CIL':30s} | SKIP (SMOTE failed)\n")

            ckpt_path = os.path.join(run_dir, f"Test_{dataset_name}_KB2_rep{rep:02d}_checkpoint.csv")
            if os.path.exists(csv_path):
                import shutil
                shutil.copy(csv_path, ckpt_path)
            logf.write(f"\n[✔] Checkpoint rep {rep}: {ckpt_path}\n")

        logf.write(f"\nKết thúc: {datetime.now()}\n")

    df_kb2 = pd.read_csv(csv_path)
    for col in ['Gmean', 'SE', 'SP', 'F1 Score', 'AUC']:
        df_kb2[col] = pd.to_numeric(df_kb2[col], errors='coerce')
    df_kb2_valid = df_kb2[~df_kb2["Name Function"].str.startswith("ERROR", na=False)]

    agg = {}
    for m in ['Gmean', 'SE', 'SP', 'F1 Score', 'AUC']:
        agg[f"{m}_mean"] = (m, 'mean')
        agg[f"{m}_std"] = (m, 'std')
    df_sum2 = df_kb2_valid.groupby("Name Method").agg(**agg).reset_index()

    sum_path = os.path.join(run_dir, f"Test_{dataset_name}_KB2_summary_{ts}.csv")
    df_sum2.to_csv(sum_path, index=False)

    print(f"\n✔ KB2 xong. CSV: {csv_path}")
    print(f"✔ TXT log   : {txt_path}")
    print(f"✔ Tổng hợp  : {sum_path}")
    print(df_sum2[["Name Method", "Gmean_mean", "Gmean_std"]].to_string(index=False))
    return csv_path, txt_path, df_sum2


csv_kb2, txt_kb2, df_kb2_sum = run_kb2(X_all, y_all, RUN_DIR)


---
## Kịch bản 3 (KB3) — Stress Test: IR Variants bằng `change_rate_data`

Tạo các biến thể của **haberman** có IR giảm dần. So sánh **3 phương pháp**:

| # | Phương pháp | Mô tả |
|---|---|---|
| 1 | **W-SVM** | Baseline |
| 2 | **AFW-CIL-fixed** | Tham số tác giả gốc (σ=1.2, K=6 cố định) |
| 3 | **BL-SMOTE + PSO-AFW-CIL** | Mô hình đề xuất |

Mục đích: chứng minh mô hình đề xuất duy trì hiệu quả khi dữ liệu rất mất cân bằng,
nơi các phương pháp cơ sở bắt đầu suy biến (G-mean → 0).

Dùng `change_rate_data(X, y, new_rate)` để lấy mẫu con lớp dương.  
CSV riêng per-IR + biểu đồ G-mean vs IR lưu vào `{RUN_DIR}/`


In [ ]:
# ── Các mức IR cần thử nghiệm ────────────────────────────────
# Haberman: 81 pos / 306 total = 26.5% → chỉ giảm được ≤ 26%
KB3_IR_LIST = [0.20, 0.15, 0.10, 0.08, 0.06, 0.04, 0.02]

def run_kb3(X_all, y_all, run_dir, dataset_name=_DS_BASE, ir_list=None):
    """
    KB3: Stress-test 3 phương pháp trên các biến thể IR giảm dần.
      1. W-SVM           — baseline
      2. AFW-CIL-fixed   — tham số gốc tác giả (σ=1.2)
      3. BL-SMOTE + PSO-AFW-CIL — mô hình đề xuất

    Tối ưu tính toán:
      - AFW-CIL: is_tomek + fuzzy_weight tính 1 lần / fold, dùng lại cho lfb_fixed
      - Proposed: PSO tính ind_posX/negX/init_weight trong __init__ (Fix C)
    Lỗi từng phương pháp được bắt riêng — không dừng toàn bộ câu.
    Ghi float('nan') vào Gm list (bỏ qua khỏi nanmean) và ERROR row vào CSV.
    Trả về: (df_summary, csv_paths_dict)
    """
    if ir_list is None:
        ir_list = KB3_IR_LIST

    ts           = datetime.now().strftime("%d%m%Y_%H%M%S")
    summary_rows = []
    csv_paths    = {}

    def _safe_auc(y_true, y_pred):
        """AUC an toàn khi test set chỉ có 1 class."""
        try:
            return roc_auc_score(y_true, y_pred)
        except ValueError:
            return float('nan')

    print(f"\n{'='*60}")
    print(f"KB3 | Dataset: {dataset_name} | IR list: {ir_list}")
    print(f"So sánh: W-SVM | AFW-CIL-fixed | BL-SMOTE+PSO-AFW-CIL")
    print(f"{'='*60}")

    for ir_target in ir_list:
        ir_pct = int(ir_target * 100)
        print(f"\n[KB3] IR = {ir_target*100:.0f}%  ({ir_target}) ...")

        # ── Tạo biến thể IR ────────────────────────────────────
        try:
            X_ir, y_ir = change_rate_data(X_all, y_all, new_rate=ir_target)
        except Exception as e:
            print(f"  [!] change_rate_data lỗi: {e} → bỏ qua IR={ir_target}")
            continue

        pos_c     = int(np.sum(y_ir == 1.0))
        neg_c     = int(np.sum(y_ir == -1.0))
        actual_ir = pos_c / len(y_ir)
        print(f"  Sau change_rate: {len(y_ir)} mẫu | +1={pos_c} | -1={neg_c} | "
              f"IR_thực={actual_ir:.4f}")

        df_ir          = pd.DataFrame(X_ir)
        df_ir['label'] = y_ir
        ckpt_ir = os.path.join(run_dir, f"Test_{dataset_name}_KB3_IR{ir_pct}pct_data.csv")
        df_ir.to_csv(ckpt_ir, index=False)

        csv_path             = os.path.join(run_dir, f"Test_{dataset_name}_KB3_IR{ir_pct}pct_{ts}.csv")
        csv_paths[ir_target] = csv_path

        min_class       = min(pos_c, neg_c)
        n_splits_actual = min(N_SPLITS, min_class)
        if n_splits_actual < 2:
            print(f"  [!] Quá ít mẫu lớp thiểu số ({min_class}) → bỏ qua toàn bộ IR này")
            continue

        # NaN (không phải 0.0) dùng cho lỗi → nanmean/nanstd sẽ bỏ qua tự động
        wsvm_gm_list, afw_gm_list, proposed_gm_list = [], [], []
        skipped_folds = 0

        for rep in range(1, N_REPEATS + 1):
            skf = StratifiedKFold(n_splits=n_splits_actual,
                                   shuffle=True, random_state=rep * 7)
            for fold, (tr_idx, te_idx) in enumerate(skf.split(X_ir, y_ir), 1):
                X_tr_raw, X_te_raw = X_ir[tr_idx], X_ir[te_idx]
                y_tr, y_te         = y_ir[tr_idx], y_ir[te_idx]

                sc   = StandardScaler()
                X_tr = sc.fit_transform(X_tr_raw)
                X_te = sc.transform(X_te_raw)

                # Guard: fold suy biến — bỏ qua toàn fold khi <2 lớp
                if len(np.unique(y_tr)) < 2 or len(np.unique(y_te)) < 2:
                    print(f"    [!] Rep {rep:02d} | Fold {fold} | <2 lớp → bỏ qua fold")
                    skipped_folds += 1
                    continue

                # ── 1. W-SVM baseline ──────────────────────────────────
                t0 = time.time()
                try:
                    pred_b = wsvm(C, X_tr, y_tr, X_te, np.ones(len(y_tr)))
                    if np.any(np.isnan(pred_b)):
                        raise ValueError("NaN in W-SVM pred")
                    sp_b, se_b, gm_b = Gmean(y_te, pred_b)
                    wsvm_gm_list.append(gm_b)
                    append_csv_row(csv_path, make_row(
                        rep, fold, round(time.time()-t0, 4),
                        "W-SVM", f"IR={ir_target}",
                        sp_b, se_b, gm_b,
                        f1_score(y_te, pred_b, pos_label=1, zero_division=0),
                        accuracy_score(y_te, pred_b), _safe_auc(y_te, pred_b),
                        str(confusion_matrix(y_te, pred_b, labels=[-1., 1.]).tolist())))
                except Exception as e:
                    print(f"    [!] W-SVM lỗi Rep {rep:02d} Fold {fold}: {e} → ghi NaN")
                    wsvm_gm_list.append(float('nan'))   # NaN: không nhiễm nanmean
                    append_csv_row(csv_path, make_row(
                        rep, fold, round(time.time()-t0, 4),
                        "W-SVM", f"ERROR,IR={ir_target}",
                        0.0, 0.0, 0.0, 0.0, 0.0, float('nan'), "error"))

                # ── 2. AFW-CIL-fixed (tác giả gốc, σ=FIXED_RATE=1.2) ──
                # is_tomek + fuzzy_weight tính 1 lần / fold → dùng lại cho lfb_fixed
                t0 = time.time()
                try:
                    _, posX_f, negX_f = is_tomek(X_tr, y_tr, class_type=[-1.0])
                    w_init_afw = fuzzy_weight(0.5, 0.8, 0.2, X_tr, y_tr,
                                             NAMEMETHOD, NAMEFUNCTION)
                    if np.any(np.isnan(w_init_afw)) or np.any(w_init_afw <= 0):
                        w_init_afw = np.ones(len(y_tr))   # fallback khi IR quá thấp
                    best_w_afw, _, _ = lfb_fixed(C, posX_f, negX_f, w_init_afw,
                                                  T_INNER, X_te, y_te, X_tr, y_tr)
                    pred_afw = wsvm(C, X_tr, y_tr, X_te, best_w_afw)
                    if np.any(np.isnan(pred_afw)):
                        raise ValueError("NaN in AFW-CIL pred")
                    sp_a, se_a, gm_a = Gmean(y_te, pred_afw)
                    afw_gm_list.append(gm_a)
                    append_csv_row(csv_path, make_row(
                        rep, fold, round(time.time()-t0, 4),
                        "AFW-CIL_fixed", f"sigma=1.2,IR={ir_target}",
                        sp_a, se_a, gm_a,
                        f1_score(y_te, pred_afw, pos_label=1, zero_division=0),
                        accuracy_score(y_te, pred_afw), _safe_auc(y_te, pred_afw),
                        str(confusion_matrix(y_te, pred_afw, labels=[-1., 1.]).tolist())))
                except Exception as e:
                    print(f"    [!] AFW-CIL-fixed lỗi Rep {rep:02d} Fold {fold}: {e} → ghi NaN")
                    afw_gm_list.append(float('nan'))    # NaN: không nhiễm nanmean
                    append_csv_row(csv_path, make_row(
                        rep, fold, round(time.time()-t0, 4),
                        "AFW-CIL_fixed", f"ERROR,IR={ir_target}",
                        0.0, 0.0, 0.0, 0.0, 0.0, float('nan'), "error"))

                # ── 3. BL-SMOTE + PSO-AFW-CIL (đề xuất) ──────────────
                # SMOTE trước → PSO_AFWCIL tính is_tomek+fuzzy_weight trong __init__
                # Fix C: dùng pso2.ind_posX / pso2.ind_negX / pso2.init_weight sau optimize()
                t0 = time.time()
                try:
                    n_pos_tr = int(np.sum(y_tr == 1.0))
                    if n_pos_tr < 2:
                        raise ValueError("minority samples < 2")
                    k_smote = min(5, n_pos_tr - 1)
                    sm               = BorderlineSMOTE(random_state=rep, k_neighbors=k_smote)
                    X_tr_sm, y_tr_sm = sm.fit_resample(X_tr, y_tr)[:2]
                    n_pos_pso2 = int(np.sum(y_tr_sm == 1.0))
                    k_cap2 = max(int(BOUNDS[0][0]), min(int(BOUNDS[0][1]), 2 * max(1, n_pos_pso2) - 1))
                    bounds_pso2 = [(int(BOUNDS[0][0]), k_cap2)] + list(BOUNDS[1:])
                    pso2 = PSO_AFWCIL(PSO_PARTICLES, PSO_ITERS, bounds_pso2, C, T_INNER,
                                      X_tr_sm, y_tr_sm, X_te, y_te,
                                      NAMEMETHOD, NAMEFUNCTION,
                                      random_state=rep * 1000 + fold + 777, warm_start=[5, 0.1, 0.5, 0.1, 0.5])
                    gbpos2, gbval2, conv2 = pso2.optimize()
                    K_b2   = pso2._normalize_k(gbpos2[0])
                    nn2_best2 = pso2._get_nn2(K_b2)
                    best_w2, _ = lfb_pso(C, pso2.ind_posX, pso2.ind_negX, pso2.init_weight,
                                          T_INNER, X_te, y_te, X_tr_sm, y_tr_sm,
                                          K_b2, gbpos2[1], gbpos2[2], gbpos2[3], gbpos2[4],
                                          nn2=nn2_best2)
                    pred_p = wsvm(C, X_tr_sm, y_tr_sm, X_te, best_w2)
                    if np.any(np.isnan(pred_p)):
                        raise ValueError("NaN in proposed pred")
                    sp_p, se_p, gm_p = Gmean(y_te, pred_p)
                    proposed_gm_list.append(gm_p)
                    fn_str2 = (f"K={K_b2},s1={gbpos2[1]:.3f},s2={gbpos2[2]:.3f},"
                               f"s3={gbpos2[3]:.3f},s4={gbpos2[4]:.3f}")
                    append_csv_row(csv_path, make_row(
                        rep, fold, round(time.time()-t0, 4),
                        "BL-SMOTE+PSO-AFW-CIL", fn_str2,
                        sp_p, se_p, gm_p,
                        f1_score(y_te, pred_p, pos_label=1, zero_division=0),
                        accuracy_score(y_te, pred_p), _safe_auc(y_te, pred_p),
                        str(confusion_matrix(y_te, pred_p, labels=[-1., 1.]).tolist())))
                except Exception as e:
                    print(f"    [!] BL-SMOTE+PSO lỗi Rep {rep:02d} Fold {fold}: {e} → ghi NaN")
                    proposed_gm_list.append(float('nan'))  # NaN: không nhiễm nanmean
                    append_csv_row(csv_path, make_row(
                        rep, fold, round(time.time()-t0, 4),
                        "BL-SMOTE+PSO-AFW-CIL", f"ERROR,IR={ir_target}",
                        0.0, 0.0, 0.0, 0.0, 0.0, float('nan'), "error"))

            print(f"  IR={ir_target*100:.0f}% | Rep {rep:02d} ✓")

        if skipped_folds > 0:
            print(f"  [i] Tổng fold bỏ qua do suy biến: {skipped_folds}")

        # nanmean/nanstd tự động bỏ qua float('nan') — không bị nhiễm bởi fold lỗi
        def _stat(lst):
            arr = [x for x in lst if x == x]   # bỏ NaN (nan != nan)
            if not arr:
                return float('nan'), 0.0
            return round(float(np.nanmean(arr)), 4), round(float(np.nanstd(arr)), 4)

        wsvm_mean, wsvm_std  = _stat(wsvm_gm_list)
        afw_mean,  afw_std   = _stat(afw_gm_list)
        prop_mean, prop_std  = _stat(proposed_gm_list)

        n_wsvm_ok  = sum(1 for x in wsvm_gm_list     if x == x)
        n_afw_ok   = sum(1 for x in afw_gm_list      if x == x)
        n_prop_ok  = sum(1 for x in proposed_gm_list  if x == x)
        n_total    = len(wsvm_gm_list)
        print(f"  Tỷ lệ thành công: W-SVM={n_wsvm_ok}/{n_total} | "
              f"AFW={n_afw_ok}/{n_total} | Proposed={n_prop_ok}/{n_total}")

        summary_rows.append({
            "IR_target":        ir_target,
            "IR_actual":        round(actual_ir, 4),
            "n_samples":        len(y_ir),
            "pos_count":        pos_c,
            "n_folds_ok":       n_total,
            "WSVM_ok":          n_wsvm_ok,
            "AFW_ok":           n_afw_ok,
            "Proposed_ok":      n_prop_ok,
            "WSVM_Gm_mean":     wsvm_mean,
            "WSVM_Gm_std":      wsvm_std,
            "AFW_Gm_mean":      afw_mean,
            "AFW_Gm_std":       afw_std,
            "Proposed_Gm_mean": prop_mean,
            "Proposed_Gm_std":  prop_std,
        })

    # ── Tổng hợp KB3 ─────────────────────────────────────────
    df_sum3  = pd.DataFrame(summary_rows)
    sum_path = os.path.join(run_dir, f"Test_{dataset_name}_KB3_summary_{ts}.csv")
    df_sum3.to_csv(sum_path, index=False)

    print(f"\n{'='*60}")
    print("BẢNG TỔNG KẾT KB3 (G-mean mean ± std, chỉ tính fold thành công)")
    cols_show = [c for c in ["IR_target", "WSVM_ok", "AFW_ok", "Proposed_ok",
                              "WSVM_Gm_mean", "AFW_Gm_mean", "Proposed_Gm_mean"] if c in df_sum3.columns]
    print(df_sum3[cols_show].to_string(index=False))

    # ── Biểu đồ G-mean vs IR (3 đường) ──────────────────────
    if len(df_sum3) > 0:
        _, ax = plt.subplots(figsize=(9, 5))
        x     = df_sum3["IR_actual"]
        ax.errorbar(x, df_sum3["WSVM_Gm_mean"],
                    yerr=df_sum3["WSVM_Gm_std"].fillna(0),
                    marker='o', label='W-SVM',
                    capsize=4, linestyle='--', color='gray')
        ax.errorbar(x, df_sum3["AFW_Gm_mean"],
                    yerr=df_sum3["AFW_Gm_std"].fillna(0),
                    marker='^', label='AFW-CIL-fixed (tác giả gốc)',
                    capsize=4, linestyle='-.', color='orange')
        ax.errorbar(x, df_sum3["Proposed_Gm_mean"],
                    yerr=df_sum3["Proposed_Gm_std"].fillna(0),
                    marker='s', label='BL-SMOTE+PSO-AFW-CIL (đề xuất)',
                    capsize=4, color='steelblue')
        ax.set_xlabel("Imbalance Ratio – tỷ lệ lớp dương (pos/total)")
        ax.set_ylabel("G-mean (mean ± std, fold lỗi bị loại)")
        ax.set_title(f"KB3: G-mean vs Imbalance Ratio — {dataset_name}")
        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.invert_xaxis()   # từ cao → thấp (stress tăng dần)
        plt.tight_layout()
        plot_path = os.path.join(run_dir, f"Test_{dataset_name}_KB3_gmean_vs_ir_{ts}.png")
        plt.savefig(plot_path, dpi=150)
        plt.show()
        print(f"✔ Biểu đồ: {plot_path}")

    print(f"✔ KB3 xong. Tổng hợp: {sum_path}")
    return df_sum3, csv_paths


# ── Chạy KB3 ─────────────────────────────────────────────────
df_kb3_sum, csv_kb3_dict = run_kb3(X_all, y_all, RUN_DIR)


---
## Tổng hợp Kết quả & Trực quan hoá

In [ ]:
def compute_average_result(csv_path, metrics=None):
    """Đọc CSV → tính mean±std theo 'Name Method'."""
    if metrics is None:
        metrics = ['SP', 'SE', 'Gmean', 'F1 Score', 'Accuracy', 'AUC']
    df = pd.read_csv(csv_path)
    for m in metrics:
        df[m] = pd.to_numeric(df[m], errors='coerce')
    agg = {}
    for m in metrics:
        agg[f"{m}_mean"] = (m, 'mean')
        agg[f"{m}_std"]  = (m, 'std')
    return df.groupby("Name Method").agg(**agg).reset_index()


def plot_comparison(summary_df, metric="Gmean",
                    title="So sánh phương pháp", save_path=None):
    """Biểu đồ cột mean ± std cho từng phương pháp."""
    mean_col = f"{metric}_mean"
    std_col  = f"{metric}_std"
    _, ax = plt.subplots(figsize=(max(8, len(summary_df) * 1.5), 5))
    colors = plt.colormaps['tab10'](np.linspace(0, 1, len(summary_df)))
    ax.bar(summary_df["Name Method"], summary_df[mean_col],
           yerr=summary_df[std_col],
           color=colors, alpha=0.85, capsize=5, edgecolor='black')
    ax.set_ylabel(f"{metric} (mean ± std)")
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f"✔ Plot saved: {save_path}")
    plt.show()


# ── Tổng hợp từ tất cả CSV của lần chạy này ──────────────────
all_csvs = sorted(glob.glob(os.path.join(RUN_DIR, "Test_*.csv")))
# Lọc bỏ checkpoint & summary & data
scenario_csvs = [f for f in all_csvs
                 if "checkpoint" not in f and "summary" not in f and "data" not in f]

print(f"\nThư mục: {RUN_DIR}")
print(f"Files CSV kịch bản:")
for f in scenario_csvs:
    print(f"  {os.path.basename(f)}")

# ── Vẽ biểu đồ cho từng kịch bản ────────────────────────────
for csv_f in scenario_csvs:
    name = os.path.basename(csv_f).replace(".csv", "")
    try:
        s = compute_average_result(csv_f)
        print(f"\n{'='*55}")
        print(f"Tổng kết: {name}")
        display(s[["Name Method", "Gmean_mean", "Gmean_std", "AUC_mean"]])
        for metric in ["Gmean", "AUC", "F1 Score"]:
            save_p = os.path.join(RUN_DIR, f"plot_{name}_{metric.replace(' ','_')}.png")
            plot_comparison(s, metric=metric,
                            title=f"{name} | {metric}",
                            save_path=save_p)
    except Exception as e:
        print(f"[!] Không thể vẽ {name}: {e}")

# ── Hiển thị tổng kết KB3 ────────────────────────────────────
if 'df_kb3_sum' in dir() and len(df_kb3_sum) > 0:
    print("\n\nKB3 — STRESS TEST SUMMARY:")
    display(df_kb3_sum)

print(f"\n✔ Tất cả kết quả trong: {RUN_DIR}")
print("Contents:")
for f in sorted(os.listdir(RUN_DIR)):
    fpath = os.path.join(RUN_DIR, f)
    size  = os.path.getsize(fpath) if os.path.isfile(fpath) else 0
    print(f"  {f:<60}  {size:>8} bytes")
